再来看一个`try...finally`的案例。

In [1]:
try:
    file = open("data.txt")
    # do something
finally:
    file.close()

NameError: name 'file' is not defined

注意，`"data.txt"`本身并不存在。

因此从一开始`try`里的`file = open("data.txt")`执行就失败了，比如文件不存在，那么 `file` 这个变量根本没有成功创建。因为没有`except`来处理错误，所以会报一个`FileNotFoundError`。

但 `finally` 还是会执行，因为它是一个【不论如何都会执行的命令】。这时它会尝试访问 `file`，结果发现 `file` 不存在，于是又报一个新的错误：`NameError: name 'file' is not defined`。

所以这个写法不安全。

通过将代码改成下面的这样，可以避免`finally`弹错误，这是一种更安全的写法。

（当然，`try`中的错误照报不误，因为没有 except。）

In [3]:
file = None

try:
    file = open("data.txt")
    # do something
finally:
    if file is not None:
        file.close()

FileNotFoundError: [Errno 2] No such file or directory: 'data.txt'

或者更常见：

In [5]:
with open("data.txt") as file:
    print("Successfully run.")

FileNotFoundError: [Errno 2] No such file or directory: 'data.txt'

注意：`with open(...) as file:` **自带 close 机制**。

它等价于一种更安全、更简洁的：

```python
file = open("data.txt")
try:
    print("Successfully run.")
finally:
    file.close()
```

也就是说：

```text
进入 with：打开文件
缩进块里面：使用文件
离开 with：自动关闭文件
```

即使 `with` 缩进块里面出错了，文件也会被关闭。

但注意：如果一开始 `open("data.txt")` 就失败，比如文件不存在，那文件根本没打开成功，也就没有东西需要 close。报错：`FileNotFoundError`。

然而这种情况下不会再报那个：`NameError: name 'file' is not defined`

因为对于这个代码来说，如果 `open("data.txt")` 一开始就失败了，程序根本不会进入 `with` 里面，也不会创建 `file`，这确实可能导致`NameError`，但因为它没有任何地方在调用 `file.close()`，所以`NameError`被触发的必要条件没有了。